# 🚀 ETL Context Example Usage

This notebook demonstrates how to:
- Create an ETL context
- Create Table Context
- Track function executions
- Query tracked data
- Use fluent API for operations
- Use TableContext display method
- Extract source information from PySpark DataFrames

## Python Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from dataframe_viewer import displayDF  # Rich formatting for DataFrames
from fluency_harness import ETLContext, TableContext

## Create Sample PySpark DataFrame

In [2]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    ArrayType, MapType, TimestampType, BooleanType
)

from datetime import datetime
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder.appName("ETLExample").getOrCreate()

# Extended schema with complex types + datetime + boolean
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("email", StringType(), True),

    # ISO3 country
    StructField("country", StringType(), True),

    # Multiple countries
    StructField("countries", ArrayType(StringType()), True),

    # Complex types
    StructField("purchases", ArrayType(StringType()), True),
    StructField("metadata", MapType(StringType(), StringType()), True),

    # Datetime column
    StructField("created_at", TimestampType(), True),

    # Boolean column
    StructField("is_active", BooleanType(), True)
])

sample_data = [
    ("C001", "John 部件", 30, "john@example.com",
     "USA", ["USA", "CAN"],
     ["laptop", "mouse"],
     {"source": "web", "tier": "gold"},
     datetime(2024, 1, 1, 10, 30),
     True),

    ("C002", "Jane Smith", 25, "jane@example.com",
     "GBR", ["GBR", "FRA"],
     ["phone"],
     {"source": "mobile", "tier": "silver"},
     datetime(2024, 1, 2, 11, 15),
     True),

    ("C003", "Bob Johnson", 35, "bob@example.com",
     "USA", ["USA"],
     ["tablet", "keyboard"],
     {"source": "store", "tier": "gold"},
     datetime(2024, 1, 3, 9, 45),
     False),

    ("C004", "Alice Brown", 28, "alice@example.com",
     "CAN", ["CAN", "NN"],
     ["monitor"],
     {"source": "web"},
     datetime(2024, 1, 4, 14, 5),
     True),

    ("C005", "Charlie Wilson 🎉", 42, "charlie@example.com",
     "USA", ["SSS", "MEX"],
     ["laptop", "headphones"],
     {"tier": "platinum"},
     datetime(2024, 1, 5, 8, 20),
     True),

    ("C006", None, 29, "david@example.com",
     "USA", ["USA"],
     ["mouse"],
     {"source": "web"},
     datetime(2024, 1, 6, 12, 0),
     False),

    ("C007", "Emma Davis", None, "emma@example.com",
     "GBR", ["GBR", "IRL"],
     None,
     {"source": "mobile"},
     datetime(2024, 1, 7, 16, 10),
     True),

    ("C008", "Frank Miller", 45, None,
     "CAN", ["CAN"],
     ["printer"],
     None,
     datetime(2024, 1, 8, 18, 25),
     None),

    ("C009", "Grace Lee", 33, "grace@example.com",
     "KOR", ["KOR", "部件"],
     ["camera", "tripod"],
     {"tier": "silver"},
     datetime(2024, 1, 9, 7, 55),
     True),

    (None, None, None, None,
     None, None,
     None,
     None,
     None,
     None),

    ("C010", "Henry White", None, None,
     "USA", ["USA"],
     ["desk"],
     {"source": "store"},
     datetime(9999, 9, 1, 13, 40),
     False),

    (None, None, None, None,
     None, None,
     None,
     None,
     None,
     None),
]

source_df = spark.createDataFrame(sample_data, schema=schema)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 01:19:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
displayDF(source_df)

customer_id,customer_name,age,email,country,countries,purchases,metadata,created_at,is_active
C001,John 部件,30,john@example.com,USA,"▶Array[2 items][ ""USA"", ""CAN"" ]","▶Array[2 items][ ""laptop"", ""mouse"" ]","▶Map[2 entries]{ ""tier"": ""gold"", ""source"": ""web"" }",2024-01-01 10:30:00,True
C002,Jane Smith,25,jane@example.com,GBR,"▶Array[2 items][ ""GBR"", ""FRA"" ]","▶Array[1 items][ ""phone"" ]","▶Map[2 entries]{ ""tier"": ""silver"", ""source"": ""mobile"" }",2024-01-02 11:15:00,True
C003,Bob Johnson,35,bob@example.com,USA,"▶Array[1 items][ ""USA"" ]","▶Array[2 items][ ""tablet"", ""keyboard"" ]","▶Map[2 entries]{ ""tier"": ""gold"", ""source"": ""store"" }",2024-01-03 09:45:00,False
C004,Alice Brown,28,alice@example.com,CAN,"▶Array[2 items][ ""CAN"", ""NN"" ]","▶Array[1 items][ ""monitor"" ]","▶Map[1 entries]{ ""source"": ""web"" }",2024-01-04 14:05:00,True
C005,Charlie Wilson 🎉,42,charlie@example.com,USA,"▶Array[2 items][ ""SSS"", ""MEX"" ]","▶Array[2 items][ ""laptop"", ""headphones"" ]","▶Map[1 entries]{ ""tier"": ""platinum"" }",2024-01-05 08:20:00,True
C006,null,29,david@example.com,USA,"▶Array[1 items][ ""USA"" ]","▶Array[1 items][ ""mouse"" ]","▶Map[1 entries]{ ""source"": ""web"" }",2024-01-06 12:00:00,False
C007,Emma Davis,null,emma@example.com,GBR,"▶Array[2 items][ ""GBR"", ""IRL"" ]",null,"▶Map[1 entries]{ ""source"": ""mobile"" }",2024-01-07 16:10:00,True
C008,Frank Miller,45,null,CAN,"▶Array[1 items][ ""CAN"" ]","▶Array[1 items][ ""printer"" ]",null,2024-01-08 18:25:00,null
C009,Grace Lee,33,grace@example.com,KOR,"▶Array[2 items][ ""KOR"", ""部件"" ]","▶Array[2 items][ ""camera"", ""tripod"" ]","▶Map[1 entries]{ ""tier"": ""silver"" }",2024-01-09 07:55:00,True
null,null,null,null,null,null,null,null,null,null


## Initialize ETL Context

In [4]:
# Create an ETL context with data product information
pipeline = ETLContext(
    #pipeline_id="pipeline_etl_001",
    source_dataset_ids=["ds_12345", "ds_source_001", "ds_source_002"],
    periodicity="Feed",
    expected_delivery="Daily",
    delivered_version="v2",
    etl_platform="AWS EMR",
    github_link="https://github.com/data-org/etl-pipeline",
    data_agreement_id="agreement_001",
    data_model_id="model_v2",
    etl_comments="Data quality checks passed. All required fields are present and validated."
)

─────────────────────────────────────────── 🚀 ETL Pipeline Initialized ───────────────────────────────────────────

 Pipeline ID           None                                                                                        
 Source Dataset IDs    ds_12345, ds_source_001, ds_source_002                                                      
 Periodicity           Feed                                                                                        
 Expected Delivery     Daily                                                                                       
 Delivered Version     v2                                                                                          
 ETL Platform          AWS EMR                                                                                     
 GitHub Link           https://github.com/data-org/etl-pipeline                                                    
 Data Agreement ID     agreement_001                                                                               
 Data Model ID         model_v2                                                                                    
 Start Time            2026-03-09 01:19:10                      

## Create Table Context with Source DataFrame

The TableContext will automatically extract source information from the DataFrame:
- Source schema (column names and data types)
- Source row count
- Source column count
- Source data quality metrics (null counts, empty row counts)

## Initialize Table Context

In [5]:
# Create table with source DataFrame (automatically extracts source schema, row count, etc.)
customer = TableContext(
    table_name="customer",
    source_df=source_df,
    stage="silver",  # Can be "bronze", "silver", or "gold",
    etl_comments="Cleaned and validated; country ISO normalized."
)

────────────────────────────────────────── Initialized 'customer' Table  ──────────────────────────────────────────

╭──── 📐 Source Dimensions ────╮                           ╭─── 📋 Source Data Quality ───╮                        
│  Column count            10  │                           │  Null value count        28  │                        
│  Row count               12  │                           │  Empty row count         2   │                        
╰──────────────────────────────╯                           ╰──────────────────────────────╯

 Column Name     Origin Type   Data Type                                  
 ───────────────────────────────────────────────────────────────────────── 
  customer_id     source        StringType()                               
  customer_name   source        StringType()                               
  age             source        IntegerType()                              
  email           source        StringType()                               
  country         source        StringType()                               
  countries       source        ArrayType(StringType(), True)              
  purchases       source        ArrayType(StringType(), True)              
  metadata        source        MapType(StringType(), StringType(), True)  
  created_at      source        TimestampType()                            
  is_active       source        BooleanType()                             

## Run Mock Transforms

In [6]:
# Run the ETL pipeline functions using fluent API
df = customer.transform.clean_data(source_df,['customer_id'])
df = customer.transform.transform_data(df, ['customer_name'])
df = customer.transform.transform_data(df, ['age'])
df = customer.transform.transform_iso(df, ['country','countries'])
df = customer.transform.transform_regex(df, ['email'])
df = customer.transform.transform_metadata(df, ['metadata'])
df = customer.transform.transform_date(df, ['purchases'])
# Run some functions multiple times
df = customer.transform.clean_data(df)  # Second run of clean_data

# Display the transformed DataFrame
customer.displayDF(source_df)

"<path fill=""currentColor"" d=""M5.50506 11.4096L6.03539 11.9399L5.50506 11.4096ZM3 14.9522H2.25H3ZM12.5904 18.4949L12.0601 17.9646L12.5904 18.4949ZM9.04776 21V21.75V21ZM11.4096 5.50506L10.8792 4.97473L11.4096 5.50506ZM13.241 17.8444C13.5339 18.1373 14.0088 18.1373 14.3017 17.8444C14.5946 17.5515 14.5946 17.0766 14.3017 16.7837L13.241 17.8444ZM7.21629 9.69832C6.9234 9.40543 6.44852 9.40543 6.15563 9.69832C5.86274 9.99122 5.86274 10.4661 6.15563 10.759L7.21629 9.69832ZM16.073 16.073C16.3659 15.7801 16.3659 15.3053 16.073 15.0124C15.7801 14.7195 15.3053 14.7195 15.0124 15.0124L16.073 16.073ZM18.4676 11.5559C18.1759 11.8499 18.1777 12.3248 18.4718 12.6165C18.7658 12.9083 19.2407 12.9064 19.5324 12.6124L18.4676 11.5559ZM6.03539 11.9399L11.9399 6.03539L10.8792 4.97473L4.97473 10.8792L6.03539 11.9399ZM6.03539 17.9646C5.18538 17.1146 4.60235 16.5293 4.22253 16.0315C3.85592 15.551 3.75 15.2411 3.75 14.9522H2.25C2.25 15.701 2.56159 16.3274 3.03 16.9414C3.48521 17.538 4.1547 18.2052 4.97473 19.0253L6.03539 17.9646ZM4.97473 10.8792C4.1547 11.6993 3.48521 12.3665 3.03 12.9631C2.56159 13.577 2.25 14.2035 2.25 14.9522H3.75C3.75 14.6633 3.85592 14.3535 4.22253 13.873C4.60235 13.3752 5.18538 12.7899 6.03539 11.9399L4.97473 10.8792ZM12.0601 17.9646C11.2101 18.8146 10.6248 19.3977 10.127 19.7775C9.64651 20.1441 9.33665 20.25 9.04776 20.25V21.75C9.79649 21.75 10.423 21.4384 11.0369 20.97C11.6335 20.5148 12.3008 19.8453 13.1208 19.0253L12.0601 17.9646ZM4.97473 19.0253C5.79476 19.8453 6.46201 20.5148 7.05863 20.97C7.67256 21.4384 8.29902 21.75 9.04776 21.75V20.25C8.75886 20.25 8.449 20.1441 7.9685 19.7775C7.47069 19.3977 6.88541 18.8146 6.03539 17.9646L4.97473 19.0253ZM17.9646 6.03539C18.8146 6.88541 19.3977 7.47069 19.7775 7.9685C20.1441 8.449 20.25 8.75886 20.25 9.04776H21.75C21.75 8.29902 21.4384 7.67256 20.97 7.05863C20.5148 6.46201 19.8453 5.79476 19.0253 4.97473L17.9646 6.03539ZM19.0253 4.97473C18.2052 4.1547 17.538 3.48521 16.9414 3.03C16.3274 2.56159 15.701 2.25 14.9522 2.25V3.75C15.2411 3.75 15.551 3.85592 16.0315 4.22253C16.5293 4.60235 17.1146 5.18538 17.9646 6.03539L19.0253 4.97473ZM11.9399 6.03539C12.7899 5.18538 13.3752 4.60235 13.873 4.22253C14.3535 3.85592 14.6633 3.75 14.9522 3.75V2.25C14.2035 2.25 13.577 2.56159 12.9631 3.03C12.3665 3.48521 11.6993 4.1547 10.8792 4.97473L11.9399 6.03539ZM14.3017 16.7837L7.21629 9.69832L6.15563 10.759L13.241 17.8444L14.3017 16.7837ZM15.0124 15.0124L12.0601 17.9646L13.1208 19.0253L16.073 16.073L15.0124 15.0124ZM19.5324 12.6124C20.1932 11.9464 20.7384 11.3759 21.114 10.8404C21.5023 10.2869 21.75 9.71511 21.75 9.04776H20.25C20.25 9.30755 20.1644 9.58207 19.886 9.979C19.5949 10.394 19.1401 10.8781 18.4676 11.5559L19.5324 12.6124Z""/>customer_id",customer_name,age,email,country,countries,purchases,"<path d=""M12.22 2h-.44a2 2 0 00-2 2v.18a2 2 0 01-1 1.73l-.43.25a2 2 0 01-2 0l-.15-.08a2 2 0 00-2.73.73l-.22.38a2 2 0 00.73 2.73l.15.1a2 2 0 011 1.72v.51a2 2 0 01-1 1.74l-.15.09a2 2 0 00-.73 2.73l.22.38a2 2 0 002.73.73l.15-.08a2 2 0 012 0l.43.25a2 2 0 011 1.73V20a2 2 0 002 2h.44a2 2 0 002-2v-.18a2 2 0 011-1.73l.43-.25a2 2 0 012 0l.15.08a2 2 0 002.73-.73l.22-.39a2 2 0 00-.73-2.73l-.15-.08a2 2 0 01-1-1.74v-.5a2 2 0 011-1.74l.15-.09a2 2 0 00.73-2.73l-.22-.38a2 2 0 00-2.73-.73l-.15.08a2 2 0 01-2 0l-.43-.25a2 2 0 01-1-1.73V4a2 2 0 00-2-2z""/>metadata",created_at,is_active
C001,John 部件,30,john@example.com,🇺🇸USA,"🇺🇸USA🇨🇦CAN▶Array[2 items][ ""USA"", ""CAN"" ]","▶Array[2 items][ ""laptop"", ""mouse"" ]","▶Map[2 entries]{ ""tier"": ""gold"", ""source"": ""web"" }",2024-01-01 10:30:00,True
C002,Jane Smith,25,jane@example.com,🇬🇧GBR,"🇬🇧GBR🇫🇷FRA▶Array[2 items][ ""GBR"", ""FRA"" ]","▶Array[1 items][ ""phone"" ]","▶Map[2 entries]{ ""tier"": ""silver"", ""source"": ""mobile"" }",2024-01-02 11:15:00,True
C003,Bob Johnson,35,bob@example.com,🇺🇸USA,"🇺🇸USA▶Array[1 items][ ""USA"" ]","▶Array[2 items][ ""tablet"", ""keyboard"" ]","▶Map[2 entries]{ ""tier"": ""gold"", ""source"": ""store"" }",2024-01-03 09:45:00,False
C004,Alice Br

## Write Table and Add to Context

In [7]:
# Add table to context for tracking
pipeline.add_table(customer)
# Write table - this sets table_type, s3_path, num_of_columns, and num_of_rows
customer.write.table(df)

──────────────────────────────────────────────── Table 1: customer ────────────────────────────────────────────────

Comments: Cleaned and validated; country ISO normalized.

 Order   Function Name        Engines         Run ID                                 Duration  
 ────────────────────────────────────────────────────────────────────────────────────────────── 
      1   clean_data           pandas v2.0.0   60def86a-e06e-4c67-9ac7-63c6a6892382   0.0000s   
      2   transform_data       spark v3.5.0    8ca641a3-1cf3-4774-a513-4f9a579d503e   0.0000s   
      3   transform_data       spark v3.5.0    9ff11e71-4eef-4342-9aba-cde2982465f0   0.0000s   
      4   transform_iso        spark v3.5.0    aa3040b4-a51e-4a00-a9d6-239bee244683   0.0000s   
      5   transform_regex      spark v3.5.0    c7831952-0914-4dfb-81cf-c9f4f9415196   0.0000s   
      6   transform_metadata   spark v3.5.0    aa2b2b0e-0b33-4b83-8e33-a45674c46393   0.0000s   
      7   transform_date       spark v3.5.0    f76fd89e-8777-4fce-abe8-79a374a0b888   0.0000s   
      8   clean_data           pandas v2.0.0   c633a9ff-f1b5-4930-b852-9f3c38469b03   0.0000s  

'table_written'

## Submit Metrics

Submit the ETL context metrics to monitoring/observability systems.

In [8]:
pipeline.submit()

╭───────────────────────────────────────────────── Pipeline Info ─────────────────────────────────────────────────╮
│  ETL Duration                                                                                                   │
│    Start                    2026-03-09 01:19:10.475612                                                          │
│    End                      2026-03-09 01:19:11.177917                                                          │
│    Duration                 0.70s                                                                               │
│                                                                                                                 │
│  Comments                                                                                                       │
│                             Data quality checks passed. All required fiel...                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Submission Status ───────────────────────────────────────────────╮
│ ✓ Metrics ready for submission                                                                                  │
│ Note: Actual submission to monitoring systems not yet implemented                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯